In [ ]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [ ]:
train_data = pd.read_csv("samsum-train.csv", engine='python')
val_data = pd.read_csv("samsum-validation.csv", engine='python')

In [ ]:
train_data

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."
...,...,...,...
13163,13809914,Deb: I missed seeing you when I visited last m...,Deb didn't meet Danie during his last visit.
13164,13730500,Caleb: Did you see the Presidents at the funer...,The presidents attended a funeral today. He di...
13165,13681152,"Martin: Hi Tom, this is my new mobile number i...",Martin and Tom are going to meet next year to ...
13166,13727605,"Tom: Hello, are you here??\r\nJerry: yes i'm\r...",Jerry is coming back home. There might be a le...


In [ ]:
train_data.shape

(13168, 3)

In [ ]:
val_data.shape

(818, 3)

In [ ]:
# random sampling
train_data = train_data.sample(n=5000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [ ]:
#data Preprocessing

import re

def clean_data(text):
  text = re.sub(r'\r\n', ' ', text) # lines
  text = re.sub(r'\s+', ' ', text) # extra space
  text = re.sub(r'<.*?>', ' ', text) #html tags
  text = text.strip().lower() # starting, ending extra spaces
  return text


In [ ]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [ ]:
train_data["dialogue"][0]

'nathan: i\'m kinda bored nathan: do you have any new anime that you can recommend? jake: obviously :) jake: just remind me what kind of anime are you interested in? nathan: something light and funny nathan: rather contemporary or fantasy settings nathan: not a big fan of sci-fi jake: did you watch anything from this fall/winter setup? nathan: not yet jake: good, that leaves us with some nice options jake: first of all probably my favorite new entry "that time i got reincarnated as a slime" jake: the title says all about it nathan: that\'s a silly name jake: well yeah but the anime is great jake: it has some darker scenes but most of the time it ticks all of your boxes jake: light, funny and fantasy setting nathan: sounds interesting jake: it is, trust me :) nathan: anything else? jake: you can try "zombieland saga" nathan: not a big fan of zombies jake: me neither jake: but that\'s an anime about girls raised as zombies to form a j-pop idol group jake: if that doesn\'t sell you on it 

In [ ]:
# tokenization (multiple token id for every single word)

tokenizer = T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [ ]:
def tokenize_function(batch):

    model_inputs = tokenizer(
        batch["dialogue"],
        max_length=512,
        padding="max_length",
        truncation=True
    )

    labels = tokenizer(
        text_target=batch["summary"],
        max_length=128,
        padding="max_length",
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [ ]:

train_dataset = train_data.apply(tokenize_function, axis=1)
val_dataset = val_data.apply(tokenize_function, axis=1)

In [ ]:
train_dataset[0]

{'input_ids': [3, 29, 9, 6736, 10, 3, 23, 31, 51, 773, 9, 19682, 3, 29, 9, 6736, 10, 103, 25, 43, 136, 126, 21705, 24, 25, 54, 1568, 58, 2662, 1050, 10, 6865, 3, 10, 61, 2662, 1050, 10, 131, 5607, 140, 125, 773, 13, 21705, 33, 25, 1638, 16, 58, 3, 29, 9, 6736, 10, 424, 659, 11, 6613, 3, 29, 9, 6736, 10, 1066, 4092, 42, 11043, 3803, 3, 29, 9, 6736, 10, 59, 3, 9, 600, 1819, 13, 17201, 18, 89, 23, 2662, 1050, 10, 410, 25, 1605, 959, 45, 48, 1590, 87, 210, 3870, 5818, 58, 3, 29, 9, 6736, 10, 59, 780, 2662, 1050, 10, 207, 6, 24, 3231, 178, 28, 128, 1245, 931, 2662, 1050, 10, 166, 13, 66, 1077, 82, 1305, 126, 1764, 96, 6279, 97, 3, 23, 530, 3, 60, 18860, 920, 38, 3, 9, 12593, 15, 121, 2662, 1050, 10, 8, 2233, 845, 66, 81, 34, 3, 29, 9, 6736, 10, 24, 31, 7, 3, 9, 17056, 564, 2662, 1050, 10, 168, 17945, 68, 8, 21705, 19, 248, 2662, 1050, 10, 34, 65, 128, 19752, 8073, 68, 167, 13, 8, 97, 34, 11331, 7, 66, 13, 39, 5598, 2662, 1050, 10, 659, 6, 6613, 11, 11043, 1898, 3, 29, 9, 6736, 10, 2993, 147

In [ ]:
#input_ids - dialogues token id

# 1 => end of sequence, 0 => paddings
# attention mask => 1 for valid input values and 0 for padding values
#labels - summary value tokens

In [ ]:
# Working With Our Model

model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
# Fine Tune

import torch

if torch.backends.mps.is_available():
  device = torch.device('mps')
else:
  device = torch.device('cpu')

print(device)

cpu


In [ ]:
# training-arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    warmup_steps=500
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.747469,0.437813
2,0.477883,0.415131
3,0.448632,0.408166
4,0.431642,0.406442
5,0.422337,0.404651
6,0.421370,0.403657


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3750, training_loss=0.881289365641276, metrics={'train_runtime': 1494.0089, 'train_samples_per_second': 20.08, 'train_steps_per_second': 2.51, 'total_flos': 4060254044160000.0, 'train_loss': 0.881289365641276, 'epoch': 6.0})

In [ ]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
# test our model

def summarize_dialogue(dialogue):
  dialogue = clean_data(dialogue)

  inputs = tokenizer(
      dialogue,
      max_length=512,
      padding="max_length",
      truncation=True,
      return_tensors="pt"
  )

  targets = model.generate(
      input_ids=inputs["input_ids"],
      attention_mask=inputs["attention_mask"],
      max_length=150,
      num_beams=4,
      early_stopping=True
  )

  summary = tokenizer.decode(targets[0], skip_special_tokens=True)
  return summary

In [ ]:
text = """A: Did you complete the machine learning assignment?
B: Not completely. I finished preprocessing and trained the model.
A: What about evaluation?
B: I still need to calculate precision, recall, and F1-score.
A: When is the deadline?
B: Tomorrow evening.
A: You should finish the evaluation today and check for overfitting.
B: Yes, I will do that tonight.
A: Also, dont forget to save the notebook and dataset.
B: Right. I will submit everything tomorrow morning."""

summary = summarize_dialogue(text)

print("summary :", summary)

summary : b finished preprocessing and trained the model. b will finish the evaluation today and check for overfitting. b will submit everything tomorrow morning.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')